In [8]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install rdkit

Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install utils

Note: you may need to restart the kernel to use updated packages.


In [28]:
pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.3 MB/s eta 0:00:00ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.9/73.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 10.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.1/111.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.3/235.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.5/231.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.5/213.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.0/349.0 kB 6.1 MB/s eta 0:00:00:00:01
Note: you may need to restart the kernel to use updated packages.


In [171]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)

cuda_available = torch.cuda.is_available()
print("CUDA available:", cuda_available)

if cuda_available:
    device = torch.cuda.get_device_properties(0)
    print("GPU name:", device.name)
    print("Total memory (GB):", round(device.total_memory / 1e9, 2))

PyTorch version: 2.7.1+cu126
CUDA version: 12.6
CUDA available: True
GPU name: NVIDIA L40
Total memory (GB): 47.58


In [14]:
from rdkit import Chem 
import numpy as np 
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init
import pandas as pd
from torch import Tensor
from torch.nn.utils.weight_norm import weight_norm
from typing import Optional
from typing import Tuple

In [150]:
# arguments for general hyperparameters, proteins, and smiles, going to args.py 
class Args:
    n_epochs   = 100
    batch_size = 64
    lr         = 0.0001
    seed       = 2048
    n_cpu      = 2
    shuffle    = True
    reg        = 5e-5
    drop       = 0.1
    S          = 'human'
    T          = 'biosnap'

    # Decoder args
    mlp_in_dim = 256
    mlp_hidden_dim = 512
    mlp_out_dim = 128
    binary = 1
args = Args()

# there are many more arguments to be added for both smiles and proteins later
class Prot_Args:
    max = 1000
    encode_dim = 512
    layers = 3
    num_heads = 8
    embedding_dim = 128
    num_filters = [128, 128, 128]
    filter_size = [3, 6, 9]
    padding = True

    # Transformer-related settings
    feed_forward_expansion_factor = 4
    feed_forward_dropout_p = 0.1
    attention_dropout_p = 0.1
    conv_dropout_p = 0.1
    conv_kernel_size = 3
    
prot_args = Prot_Args()

class Smiles_Args:
    max_nodes = 290
    drug_in = 78 # big question 
    embedding_dim =  128
    hidden_layers = 128
    padding = True
    
smiles_args = Smiles_Args()

In [151]:
import os 
from pathlib import Path
cwd = Path.cwd()
print(cwd)

/home/jovyan/DTI-SL


In [152]:
import utils
from functools import partial
from torch.utils.data import Dataset

In [153]:
import torch.nn.functional as F
from torch import Tensor
from typing import Optional

In [154]:
from rdkit import Chem
from rdkit import RDLogger
import numpy as np
import torch

# Optional: suppress RDKit warnings (e.g. about deprecated C++ internals)
RDLogger.DisableLog('rdApp.warning')

def get_atom_features(atom):
    features = []

    # One-hot atomic number (up to 64 elements) — reduce from 100 to 64
    atomic_number = atom.GetAtomicNum()
    atomic_onehot = np.zeros(64, dtype=np.float32)
    if atomic_number < 64:
        atomic_onehot[atomic_number] = 1.0
    features.extend(atomic_onehot)

    # Standard features (6)
    features.append(atom.GetDegree())
    features.append(atom.GetTotalNumHs())
    features.append(atom.GetImplicitValence())
    features.append(atom.GetFormalCharge())
    features.append(atom.GetNumRadicalElectrons())
    features.append(int(atom.GetIsAromatic()))

    # Hybridization one-hot (5)
    hybridization_types = [
        Chem.rdchem.HybridizationType.SP,
        Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3,
        Chem.rdchem.HybridizationType.SP3D,
        Chem.rdchem.HybridizationType.SP3D2
    ]
    hybridization_onehot = [int(atom.GetHybridization() == h) for h in hybridization_types]
    features.extend(hybridization_onehot)

    # Chirality (2)
    features.append(int(atom.HasProp('_ChiralityPossible')))
    features.append(int(atom.GetChiralTag() != Chem.rdchem.ChiralType.CHI_UNSPECIFIED))

    return np.array(features, dtype=np.float32)

def get_bond_features(bond):
    bt = bond.GetBondType()
    features = [
        int(bt == Chem.rdchem.BondType.SINGLE),
        int(bt == Chem.rdchem.BondType.DOUBLE),
        int(bt == Chem.rdchem.BondType.TRIPLE),
        int(bt == Chem.rdchem.BondType.AROMATIC),
        int(bond.GetIsConjugated()),
        int(bond.IsInRing()),
        int(bond.GetStereo() != Chem.rdchem.BondStereo.STEREONONE)
    ]
    return np.array(features, dtype=np.float32)

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES string")

    atom_features = []
    edge_indices = []
    edge_features = []

    for atom in mol.GetAtoms():
        atom_features.append(get_atom_features(atom))

    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        bond_feat = get_bond_features(bond)

        edge_indices += [(i, j), (j, i)]
        edge_features += [bond_feat, bond_feat]

    x = torch.tensor(atom_features, dtype=torch.float)
    edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous() if edge_indices else torch.empty((2, 0), dtype=torch.long)
    edge_attr = torch.tensor(edge_features, dtype=torch.float) if edge_features else torch.empty((0, 7), dtype=torch.float)

    return x, edge_index, edge_attr


In [155]:
# setting up the protein matrix, going to loader.py
# ---------- amino-acid dictionary ----------
amino_dict = {"B": 1, "A": 2, "C": 3, "E": 4, "D": 5, "G": 6,
              "F": 7, "I": 8, "H": 9, "K": 10, "M": 11, "L": 12,
              "O": 13, "N": 14, "Q": 15, "P": 16, "S": 17, "R": 18,
              "U": 19, "T": 20, "W": 21, "V": 22, "X": 23, "Z": 24, "Y": 25}

# ---------- helper: encode one protein ----------
def encode_protein(seq: str, max_len: int):
    encoded = [amino_dict.get(res, 0) for res in seq[:max_len]]
    if len(encoded) < max_len:
        encoded.extend([0] * (max_len - len(encoded)))
    return np.asarray(encoded, dtype=np.int16)

In [156]:
# data class, uses functions above^

class DrugProteinDataset(Dataset):
    """
    Dataset for drug-protein pairs:
    • Encodes protein sequences once (as int tensors)
    • Builds molecule graphs on-demand with padded nodes
    • Returns (graph_dict, protein_tensor, mask, label)
    """
    def __init__(self, df, prot_args, smiles_args):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.p_max = prot_args.max                   # e.g., 1000 amino acids
        self.d_max = smiles_args.max_nodes           # e.g., 290 max atoms

        # Pre-encode proteins
        self.protein_int = np.stack(
            df["Protein"].apply(lambda s: encode_protein(s, self.p_max)).values
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        smiles = row["SMILES"]

        # --- Build molecule graph ---
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError(f"Invalid SMILES at idx {idx}")

        # Real atom features with mask bit = 0
        atom_feats = [np.concatenate([get_atom_features(atom), [0.0]]) for atom in mol.GetAtoms()]
        atom_feats = np.stack(atom_feats, axis=0)  # [n_real, feat_dim]
        n_real = atom_feats.shape[0]

        if n_real > self.d_max:
            raise ValueError(f"SMILES at idx {idx} has {n_real} atoms > max_nodes={self.d_max}")

        # Virtual atoms: match feature size + set mask bit = 1
        n_fake = self.d_max - n_real
        if n_fake > 0:
            feat_dim = atom_feats.shape[1]  ## e.g., 77 + 1 = 78 including mask bit
            virtual_feats = np.zeros((n_fake, feat_dim), dtype=np.float32)
            virtual_feats[:, -1] = 1.0  # mask bit set to 1 for padded atoms
            atom_feats = np.concatenate([atom_feats, virtual_feats], axis=0)

        x = torch.from_numpy(atom_feats).float()  # shape: [d_max, feat_dim]

        # Edges (optional)
        edge_index = []
        edge_attr = []
        for bond in mol.GetBonds():
            i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            feat = get_bond_features(bond)  # you can keep your existing get_bond_features()
            edge_index += [[i, j], [j, i]]
            edge_attr += [feat, feat]

        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous() if edge_index else torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.tensor(edge_attr, dtype=torch.float32) if edge_attr else torch.empty((0, 7), dtype=torch.float32)

        g = {
            'x': x,                      # [d_max, feat_dim]
            'edge_index': edge_index,   # [2, num_edges]
            'edge_attr': edge_attr      # [num_edges, 7]
        }

        # --- Protein ---
        prot_int = torch.tensor(self.protein_int[idx], dtype=torch.long)  # [p_max]
        mask = (prot_int != 0).float()                                    # [p_max]

        # --- Label ---
        y = torch.tensor(row["Y"], dtype=torch.float32)

        return g, prot_int, mask, y


In [157]:
# home/jovyan is from JupyterHub, going to loader.py
S_dir = Path("/home/jovyan/DTI-SL/datasets") / args.S # domain here

train = pd.read_csv(S_dir / "train.csv")[:500]
val   = pd.read_csv(S_dir / "val.csv")[:100]
test  = pd.read_csv(S_dir / "test.csv")[:100]

In [158]:
train.shape, val.shape, test.shape, train.columns, train.head() # 3 for drug,protein,label 

((500, 3),
 (100, 3),
 (100, 3),
 Index(['SMILES', 'Protein', 'Y'], dtype='object'),
                                               SMILES  \
 0  CC(C)[C@H](NS(=O)(=O)C1=CC=C(C=C1)C2=CC=C(C=C2...   
 1  [O-2].[O-2].[O-2].[O-2].[O-2].[O-2].[O-2].[O-2...   
 2              CC12CCC3C(C1CCC2=O)CCC4=CC(=O)CCC34CO   
 3  C1CN(CCC1(C2=CC=C(C=C2)Cl)O)CCCC(=O)C3=CC=C(C=...   
 4  C1C[C@@H]2CN[C@H](C[C@@H]2C[C@@H]1CCC3=NNN=N3)...   
 
                                              Protein  Y  
 0  MILLTFSTGRRLDFVHHSGVFFLQTLLWILCATVCGTEQYFNVEVW...  1  
 1  MLPSASRERPGYRAGVAAPDLLDPKSAAQNSKPRLSFSTKPTVLAS...  0  
 2  MWSWKCLLFWAVLVTATLCTARPSPTLPEQAQPWGAPVEVESFLVH...  0  
 3  MPVRRGHVAPQNTFLDTIIRKFEGQSRKFIIANARVENCAVIYCND...  1  
 4  MAEDGEEAEFHFAALYISGQWPRLRADTDLQRLGSSAMAPSRKFFV...  0  )

In [196]:
# test block  NEED CHANGE 
prot_args   = Prot_Args()
smiles_args = Smiles_Args()

train_data = DrugProteinDataset(train, prot_args, smiles_args)
val_data   = DrugProteinDataset(val,   prot_args, smiles_args)
test_data  = DrugProteinDataset(test,  prot_args, smiles_args)

# sanity check, this will show nodes, edges, protein size for a single instance 
g, prot_int, mask, y = train_data[40]
print(g['x'].shape)         # e.g., torch.Size([290, 75])
print(g['edge_index'].shape)
print(g['edge_attr'].shape)
print(prot_int.size(), mask.sum(), y)

torch.Size([290, 78])
torch.Size([2, 66])
torch.Size([66, 7])
torch.Size([1000]) tensor(977.) tensor(0.)


In [183]:
from torch_geometric.nn import GCNConv
from torch_geometric.data import Batch

class DrugGCN(nn.Module):
    def __init__(self, in_feats,
                 dim_embedding=128, 
                 padding=True,
                 hidden_feats=None,
                 activation=None):
        super(DrugGCN, self).__init__()
    
        self.init_transform = nn.Linear(in_feats, dim_embedding, bias=False)
    
        if padding:
            with torch.no_grad():
                self.init_transform.weight[-1].fill_(0)
    
        if hidden_feats is None:
            hidden_feats = [dim_embedding] * 3

        # trouble here
        self.activation = nn.ReLU()
    
        self.convs = nn.ModuleList()
        prev_dim = dim_embedding
        for h_dim in hidden_feats:
            self.convs.append(GCNConv(prev_dim, h_dim))
            prev_dim = h_dim
    
        self.output_feats = hidden_feats[-1]

    def forward(self, data):
        # data: PyG Batch object with x (node features) and edge_index (graph structure)
        x, edge_index, batch = data.x, data.edge_index, data.batch
        print(x.shape, edge_index.shape)
        x = self.init_transform(x)

        for conv in self.convs:
            x = conv(x, edge_index)
            x = self.activation(x)

        # Reshape to [batch_size, num_nodes_per_graph, output_feats]
        # You may need padding or pooling if graphs have variable sizes
        # Here we use global mean pooling as an example
        from torch_geometric.nn import global_mean_pool
        out = global_mean_pool(x, batch)  # [batch_size, output_feats]

        return out

In [184]:
# CNNTrans, protein encoder block # SHOULD BE OKAY

class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)


class GLU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        a, b = x.chunk(2, dim=self.dim)
        return a * torch.sigmoid(b)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=10000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return self.pe[:, :x.size(1)]


class RelativeMultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.pos_proj = nn.Linear(d_model, d_model, bias=False)
        self.u_bias = nn.Parameter(torch.randn(num_heads, self.d_head))
        self.v_bias = nn.Parameter(torch.randn(num_heads, self.d_head))
        self.out_proj = nn.Linear(d_model, d_model)
        self.scale = math.sqrt(d_model)

    def forward(self, x, pos_enc, mask=None):
        B, L, _ = x.size()
        q = self.q_proj(x).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        p = self.pos_proj(pos_enc).view(B, L, self.num_heads, self.d_head).transpose(1, 2)

        content_score = torch.matmul(q + self.u_bias.unsqueeze(1), k.transpose(-2, -1))
        pos_score = torch.matmul(q + self.v_bias.unsqueeze(1), p.transpose(-2, -1))
        score = (content_score + self._relative_shift(pos_score)) / self.scale

        if mask is not None:
            score = score.masked_fill(mask.unsqueeze(1).unsqueeze(2) == 0, -1e9)

        attn = F.softmax(score, dim=-1)
        context = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, L, -1)
        return self.out_proj(context)

    def _relative_shift(self, x):
        B, H, L1, L2 = x.size()
        zero_pad = torch.zeros((B, H, L1, 1), device=x.device, dtype=x.dtype)
        x_padded = torch.cat([zero_pad, x], dim=-1)
        x_padded = x_padded.view(B, H, L2 + 1, L1)
        return x_padded[:, :, 1:].view(B, H, L1, L2)


class FeedForwardModule(nn.Module):
    def __init__(self, d_model, expansion=4, dropout=0.1):
        super().__init__()
        self.seq = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model * expansion),
            Swish(),
            nn.Dropout(dropout),
            nn.Linear(d_model * expansion, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.seq(x)


class ConvModule(nn.Module):
    def __init__(self, d_model, kernel_size=5, dropout=0.1):
        super().__init__()
        self.seq = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Conv1d(d_model, d_model * 2, kernel_size=1),
            Swish(),
            nn.Conv1d(d_model * 2, d_model * 2, kernel_size, padding=(kernel_size - 1) // 2),
            GLU(dim=1),
            nn.BatchNorm1d(d_model),
            nn.Conv1d(d_model, d_model, kernel_size=1),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.seq(x)
        return x.transpose(1, 2)


class CNNTransBlock(nn.Module):
    def __init__(self, d_model=512, num_heads=8, kernel_size=5, dropout=0.1, max_len=1000):
        super().__init__()
        self.attn = RelativeMultiHeadAttention(d_model, num_heads)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.norm = nn.LayerNorm(d_model)
        self.ff = FeedForwardModule(d_model, dropout=dropout)
        self.conv = ConvModule(d_model, kernel_size, dropout)

    def forward(self, x, mask):
        pos = self.pos_enc(x)
        x = self.attn(self.norm(x), pos, mask) + x
        x = self.conv(x) + x
        return 0.5 * self.ff(x) + 0.5 * x


class ProteinCNNTrans(nn.Module):
    def __init__(self, max_len=1000, encoder_dim=128, num_layers=3, **kwargs):
        super().__init__()
        self.blocks = nn.ModuleList([
            CNNTransBlock(d_model=encoder_dim, max_len=max_len, **kwargs)
            for _ in range(num_layers)
        ])
        self.final_ff = FeedForwardModule(encoder_dim)

    def forward(self, x, mask):
        for block in self.blocks:
            x = block(x, mask)
        return 0.5 * self.final_ff(x) + 0.5 * x

In [185]:
# decoder block  SHOULD BE OKAY 
class MLPClassifier(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, binary=1):
        super(MLPClassifier, self).__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, out_dim)
        self.bn3 = nn.BatchNorm1d(out_dim)
        self.fc4 = nn.Linear(out_dim, binary)

    def forward(self, x):
        x = self.bn1(F.relu(self.fc1(x)))
        x = self.bn2(F.relu(self.fc2(x)))
        x = self.bn3(F.relu(self.fc3(x)))
        x = self.fc4(x)
        return x

In [186]:
prot_args = Prot_Args()
smiles_args = Smiles_Args()
args = Args()
# NEED CHANGE 
class DTIModel(nn.Module):
    def __init__(self, smiles_args, prot_args, **config):
        super(DTIModel, self).__init__()

        self.drug_in = smiles_args.drug_in
        self.embedding_dim = smiles_args.embedding_dim
        self.max_nodes = smiles_args.max_nodes
        self.padding = smiles_args.padding
        self.hidden_layers = smiles_args.hidden_layers 

         # Drug feature extractor using Graph Convolutional Network (GCN)
        self.drug_encoder = DrugGCN(
            in_feats=self.drug_in,
            dim_embedding=self.embedding_dim,
            hidden_feats=[self.hidden_layers] * 3, # three layers GCN 128x128x128
            activation=[F.relu] * 3,
            padding=self.padding
        )

        # Protein feature encoder combining CNN and Transformer
        self.protein_encoder = ProteinCNNTrans(
            max_len=prot_args.max,
            encoder_dim=prot_args.embedding_dim,
            num_layers=prot_args.layers,
            num_heads=prot_args.num_heads,
            kernel_size=prot_args.conv_kernel_size,
            dropout=prot_args.conv_dropout_p
        )
        
         # MLP decoder for the final classification output
        self.mlp_classifier = MLPClassifier(
            args.mlp_in_dim,  # Input dimension for MLP
            args.mlp_hidden_dim,  # Hidden layer dimension for MLP
            args.mlp_out_dim,  # Output dimension for MLP
            args.binary  # Binary classification flag for MLP output
        )

        self.protein_embed = nn.Embedding(26, 128, padding_idx=0)  # 26 possible amino acids, embedded into 128-dimensional space

        self.mix_attention_layer = nn.MultiheadAttention(128, 4)  # 128-dimensional input, 4 attention heads

        # Max pooling layers for drug and protein feature extraction
        self.Drug_max_pool = nn.MaxPool1d(290)  # Pooling layer for drug features (max pool over 290 values)
        self.Protein_max_pool = nn.MaxPool1d(1000)  # Pooling layer for protein features (max pool over 1000 values)

        # Dropout layer for regularization
        self.dropout1 = nn.Dropout(0.1)  # Dropout with a probability of 10%

    def forward(self, bg_d, v_p, protein_mask, mode="train"):
        # Process drug graph through molecular GCN feature extractor
        v_d = self.drug_encoder(bg_d)
        
        # Embed protein sequences using the protein embedding layer
        v_p = self.protein_embed(v_p.long().to(device))  # Convert protein indices to embeddings
        protein_mask = protein_mask.long().to(device)  # Protein mask for attention
        
        # Process protein embeddings through the protein feature encoder
        v_p = self.protein_encoder(v_p, protein_mask)

        # Prepare for attention by permuting the dimensions of drug and protein features
        drugConv = v_d.permute(0, 2, 1)  # Permute for attention processing
        proteinConv = v_p.permute(0, 2, 1)  # Permute for attention processing
        
        # Prepare drug and protein for attention mechanism (Q, K, V are query, key, value)
        drug_QKV = drugConv.permute(2, 0, 1)  # Query for drug
        protein_QKV = proteinConv.permute(2, 0, 1)  # Query for protein
        
        # Apply multi-head attention between drug and protein
        drug_att, _ = self.mix_attention_layer(drug_QKV, protein_QKV, protein_QKV)
        protein_att, _ = self.mix_attention_layer(protein_QKV, drug_QKV, drug_QKV)

        # Permute back after attention to get the correct shape
        drug_att = drug_att.permute(1, 2, 0)
        protein_att = protein_att.permute(1, 2, 0)

        # Combine original and attended features
        drugConv = drugConv * 0.5 + drug_att * 0.5
        proteinConv = proteinConv * 0.5 + protein_att * 0.5

        # Apply max pooling to both drug and protein features
        drugConv = self.Drug_max_pool(drugConv).squeeze(2)
        proteinConv = self.Protein_max_pool(proteinConv).squeeze(2)

        # Concatenate drug and protein attended features and apply dropout
        result = torch.cat((drug_att, protein_att), dim=-1)
        pair = torch.cat([drugConv, proteinConv], dim=1)
        pair = self.dropout1(pair)

        # Pass concatenated features through MLP classifier
        score = self.mlp_classifier(pair)

        # Return results based on mode (train or eval)
        if mode == "train":
            return v_d, v_p, pair, score  # Return all outputs for training
        elif mode == "eval":
            return v_d, v_p, score, result  # Return simplified outputs for evaluation
        
            

In [187]:
import random
from torch.utils.data import DataLoader
import math
args = Args()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

set_seed(args.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [188]:
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

def custom_collate(batch):
    graphs, proteins, masks, labels = zip(*batch)

    xs = []
    edge_indices = []
    edge_attrs = []
    node_offset = 0
    batch_vector = []  # NEW: which graph each node belongs to

    for i, g in enumerate(graphs):
        x = g['x']
        xs.append(x)

        ei = g['edge_index']
        ea = g['edge_attr']
        if ei.numel() > 0:
            edge_indices.append(ei + node_offset)
            edge_attrs.append(ea)

        # Append graph index for each node in this graph
        batch_vector.append(torch.full((x.shape[0],), i, dtype=torch.long))

        node_offset += x.shape[0]

    x_batch = torch.cat(xs, dim=0)
    if edge_indices:
        edge_index_batch = torch.cat(edge_indices, dim=1)
        edge_attr_batch = torch.cat(edge_attrs, dim=0)
    else:
        edge_index_batch = torch.empty((2, 0), dtype=torch.long)
        edge_attr_batch = torch.empty((0, 13), dtype=torch.float32)

    batch_vector = torch.cat(batch_vector, dim=0)  # [total_nodes]

    batch_graph = {
        'x': x_batch,
        'edge_index': edge_index_batch,
        'edge_attr': edge_attr_batch,
        'batch': batch_vector,  # NEW
    }

    proteins = torch.stack(proteins, dim=0)
    masks = torch.stack(masks, dim=0)
    labels = torch.stack(labels, dim=0)

    return batch_graph, proteins, masks, labels


In [195]:
S_dir = Path("/home/jovyan/DTI-SL/datasets") / args.S

train_df = pd.read_csv(S_dir / "train.csv")
val_df   = pd.read_csv(S_dir / "val.csv")
test_df  = pd.read_csv(S_dir / "test.csv")

train_data = DrugProteinDataset(train_df, prot_args, smiles_args)
val_data   = DrugProteinDataset(val_df, prot_args, smiles_args)
test_data  = DrugProteinDataset(test_df, prot_args, smiles_args)
print(len(train_data), len(val_data), len(test_data))
train_loader = DataLoader(train_data, batch_size=args.batch_size, collate_fn=custom_collate)
val_loader   = DataLoader(val_data, batch_size=args.batch_size, collate_fn=custom_collate)
test_loader  = DataLoader(test_data, batch_size=args.batch_size, collate_fn=custom_collate)
batch = next(iter(train_loader))
g, prot, mask, y = batch

print("Drug graph x:", g['x'].shape)              # [total_nodes, feat_dim]
print("Drug edge_index:", g['edge_index'].shape)  # [2, total_edges]
print("Drug edge_attr:", g['edge_attr'].shape)    # [total_edges, 7]

print("Protein sequences:", prot.shape)           # [batch_size, p_max]
print("Protein masks:", mask.shape)               # [batch_size, p_max]
print("Labels:", y.shape)        

4197 600 1200
Drug graph x: torch.Size([18560, 78])
Drug edge_index: torch.Size([2, 2948])
Drug edge_attr: torch.Size([2948, 7])
Protein sequences: torch.Size([64, 1000])
Protein masks: torch.Size([64, 1000])
Labels: torch.Size([64])


In [190]:
model = DTIModel(smiles_args, prot_args).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.reg)
criterion = torch.nn.BCEWithLogitsLoss()


In [191]:
from torch_geometric.data import Data

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    total = 0
    correct = 0

    for batch in loader:
        g, v_p, mask, y = batch
        x = g['x'].to(device)
        edge_index = g['edge_index'].to(device)
        edge_attr = g['edge_attr'].to(device)
        batch_vector = g['batch'].to(device)  # NEW

        v_p = v_p.to(device)
        mask = mask.to(device)
        y = y.to(device)

        batch_graph = Data(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            batch=batch_vector,  # NEW
        )

        optimizer.zero_grad()
        _, _, _, score = model(batch_graph, v_p, mask, mode="train")
        loss = criterion(score.view(-1), y.view(-1))
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)

        preds = (torch.sigmoid(score.view(-1)) > 0.5).float()
        correct += (preds == y).sum().item()
        total += y.size(0)

    acc = correct / total
    avg_loss = running_loss / total
    return avg_loss, acc


In [192]:
# val
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    total = 0
    correct = 0

    for batch in loader:
        g, v_p, mask, y = batch
        x = g['x'].to(device)
        edge_index = g['edge_index'].to(device)
        edge_attr = g['edge_attr'].to(device)
        v_p = v_p.to(device)
        mask = mask.to(device)
        y = y.to(device)

        # Convert dict to PyG Data object
        batch_graph = Data(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr
        )

        _, _, score, _ = model(batch_graph, v_p, mask, mode="eval")
        loss = criterion(score.view(-1), y.view(-1))

        running_loss += loss.item() * y.size(0)
        preds = (torch.sigmoid(score.view(-1)) > 0.5).float()
        correct += (preds == y).sum().item()
        total += y.size(0)

    acc = correct / total
    avg_loss = running_loss / total
    return avg_loss, acc

In [193]:
# train and val 
num_epochs = 10

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}  | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f}  | Val   Acc: {val_acc:.4f}")


torch.Size([18560, 78]) torch.Size([2, 2948])


RuntimeError: shape '[64, 1000, 8, 16]' is invalid for input of size 128000

In [194]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test  Loss: {test_loss:.4f}  | Test  Acc: {test_acc:.4f}")


torch.Size([18560, 78]) torch.Size([2, 2536])


RuntimeError: shape '[64, 1000, 8, 16]' is invalid for input of size 128000